# KOSPI200 공매도 압력 가설 검정

이 노트북은 3주차 과제 PDF의 레시피에 맞춰 공매도 압력비율을 만들고, 고압력 D10의 다음 달 EW/VW 언더퍼폼을 검정합니다.

- 유니버스: 월말 KOSPI200 편입 종목
- 주 프록시: 대차거래잔고수량
- 보조 프록시: 차입공매도잔고수량
- 분모: 시가총액(백만원) * 1,000,000 / 원종가
- 수익률: 수정주가의 다음 달 수익률


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name in ('code', 'notebooks', 'src'):
    ROOT = ROOT.parent
CODE_DIR = ROOT / 'src'
OUTPUT_DIR = ROOT / 'output' / 'tables'
PROCESSED_DIR = ROOT / 'input' / 'processed_data'
sys.path.insert(0, str(CODE_DIR))

import short_pressure_analysis as spa

ROOT


PosixPath('kospi200-short-pressure-test')

## 1. 데이터 수집 항목 확인

원본 Excel에는 PDF가 요구한 가격, 유니버스, 시가총액, 대차잔고, 차입공매도잔고 항목이 들어 있습니다. 차입공매도잔고수량은 일별 행으로 들어 있으므로 월별 마지막 관측치를 사용합니다.


In [2]:
from openpyxl import load_workbook

wb = load_workbook(spa.RAW_XLSX, read_only=True, data_only=True)
meta = []
for ws in wb.worksheets:
    meta.append({
        'sheet': ws.title,
        'rows': ws.max_row,
        'cols': ws.max_column,
        'output_frequency': ws['B5'].value,
        'start': ws['B8'].value,
        'end': ws['C8'].value,
        'item': ws['B13'].value,
    })
pd.DataFrame(meta)


,sheet,rows,cols,output_frequency,start,end,item
0,종가,212,3981,월간,20100101,20260630,종가(원)
1,수정주가,212,3981,월간,20100101,20260630,수정주가(원)
2,시가총액,212,3981,월간,20100101,20260630,시가총액(백만원)
3,코스피200구성여부,212,3981,월간,20100101,20260630,"코스피200 구성종목여부(1:해당, 0:미해당)"
4,대차거래잔고수량,212,3981,월간,20100101,20260630,대차거래잔고수량(주)
5,차입공매도잔고수량,4073,3981,월간,20100101,20260630,차입공매도잔고수량(주)


## 2. 전처리와 검정 실행

`short_pressure_analysis.py`는 다음 처리를 수행합니다.

1. 시가총액을 원 단위로 환산한 뒤 원종가로 나누어 상장주식수를 역산합니다.
2. 직전월 대비 역산 상장주식수가 3배 초과로 변하면 해당 종목-월을 결측 처리합니다.
3. 잔고수량 / 상장주식수 * 100으로 압력비율을 만듭니다.
4. 공매도 금지기간이 신호월 또는 다음 달 수익률 측정월에 걸리면 제외합니다.
5. 매월 KOSPI200 내 압력비율 상위 10% D10의 다음 달 수익률에서 유니버스 평균을 차감합니다.
6. EW/VW 시계열의 연환산 평균과 t값을 계산합니다.


In [3]:
# 결과 파일을 재생성합니다. 대용량 Excel을 읽기 때문에 약 40~60초 걸릴 수 있습니다.
spa.main()


Created:
processed_data/universe_quality_by_month.csv
processed_data/next_month_returns.csv
processed_data/pressure_ratio_short_balance.csv
processed_data/pressure_ratio_lending_balance.csv
processed_data/analysis_panel_short_balance.csv
processed_data/analysis_panel_lending_balance.csv
output/monthly_underperformance.csv
output/proxy_test_summary.csv
output/d10_constituents.csv
output/d10_max_weight_check.csv
output/cumulative_underperformance.svg
output/cumulative_underperformance.png
output/final_judgement_draft.md


## 3. 필수 검정 결과

PDF의 판정표 입력값인 `t_EW`, `t_VW`를 확인합니다.


In [4]:
summary = pd.read_csv(OUTPUT_DIR / 'proxy_test_summary.csv')
summary


,proxy,first_period,last_period,n_months,ann_mean_ew,t_ew,ann_mean_vw,t_vw,judgement,avg_universe_count,avg_reported_balance_count,avg_nonzero_balance_count,months_d10_max_weight_over_50pct,max_d10_max_weight
0,차입공매도잔고수량,2016-06,2026-05,87,0.126701,2.821822,-0.052206,-1.162051,신호 없음 / 결론유보,200.034483,33.873563,33.770115,4,0.565829
1,대차거래잔고수량,2010-01,2026-05,159,-0.018098,-0.518949,-0.122115,-2.418677,A+ - VW 중심 유의,199.572327,199.540881,199.509434,0,0.425683


In [5]:
monthly = pd.read_csv(OUTPUT_DIR / 'monthly_underperformance.csv')
monthly.groupby('proxy')[['u_ew', 'u_vw', 'd10_max_weight']].agg(['count', 'mean', 'std', 'max'])


u_ew                                u_vw                      \
          count      mean       std       max count      mean       std   
proxy                                                                     
대차거래잔고수량    159 -0.001508  0.036646  0.145613   159 -0.010176  0.053053   
차입공매도잔고수량    87  0.010558  0.034900  0.151360    87 -0.004351  0.034920   

                    d10_max_weight                                
                max          count      mean       std       max  
proxy                                                             
대차거래잔고수량   0.220056            159  0.252668  0.081764  0.425683  
차입공매도잔고수량  0.058590             87  0.185450  0.101215  0.565829

## 4. VW 집중도 점검

D10의 단일종목 최대 비중이 50%를 넘는 달이 잦으면 VW 결과를 한 종목 베팅으로 해석해야 합니다.


In [6]:
weight_check = pd.read_csv(OUTPUT_DIR / 'd10_max_weight_check.csv')
weight_check.groupby('proxy').agg(
    months=('period', 'count'),
    months_over_50pct=('d10_max_weight', lambda s: (s > 0.5).sum()),
    max_weight=('d10_max_weight', 'max'),
    avg_weight=('d10_max_weight', 'mean'),
)


,months,months_over_50pct,max_weight,avg_weight
proxy,,,,
대차거래잔고수량,159,0,0.425683,0.252668
차입공매도잔고수량,87,4,0.565829,0.185450


## 5. 시각화

아래 그래프는 고압력 D10 언더퍼폼 시계열을 누적한 참고 시각화입니다.

![cumulative_underperformance](../output/cumulative_underperformance.png)


## 6. 판정

주 프록시인 대차거래잔고수량 기준으로 EW는 비유의이고 VW는 유의합니다. PDF 판정표 기준으로는 `A+ - 대형주 주도`에 해당하므로, 본 표본과 프록시에서는 가설 A가 조건부로 더 타당하다고 해석합니다. 다만 대차잔고는 실제 공매도 포지션만이 아니라는 점과, 차입공매도잔고수량은 커버리지가 낮다는 점을 한계로 둡니다.
